In [ ]:
"""
Combine 4 GPKG files into one 'infrastructure.gpkg' with layers 'reseller_africa'
and 'filling_africa'. Points falling outside African countries are moved to the
nearest interior point, but only if the displacement is ≤ 2 km. Points that would
need a larger move are removed. Duplicate points within each category (reseller,
filling) are deduplicated. Overlapping points between categories are separated by
2km to avoid perfect coincidence.
"""

import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from shapely.ops import nearest_points, unary_union

# ------------------------------------------------------------
# 1. CONFIGURE RELATIVE PATHS HERE
# ------------------------------------------------------------
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()
    print(f"Using current working directory as base: {SCRIPT_DIR}")

PATHS = {
    "reseller1": os.path.join(SCRIPT_DIR, "SSA/output_africa_BigCity/all_africa_combined_20260429_104943_reseller_3857.gpkg"),
    "reseller2": os.path.join(SCRIPT_DIR, "SSA/location_total/all_africa_combined_reseller_3857.gpkg"),
    "filling1": os.path.join(SCRIPT_DIR, "SSA/output_africa_BigCity/all_africa_combined_20260429_104943_filling_3857.gpkg"),
    "filling2": os.path.join(SCRIPT_DIR, "SSA/location_total/all_africa_combined_filling_3857.gpkg"),
    "countries": os.path.join(SCRIPT_DIR, "SSA/location_total/world_bound/ne_110m_admin_0_countries.shp"),
    "output": os.path.join(SCRIPT_DIR, "SSA/infrastructure.gpkg"),
}

MAX_MOVE_KM = 2          # maximum displacement in kilometres
SEPARATION_M = 2         # shift filling point if coincident with a reseller point
# ------------------------------------------------------------


def move_inside(point, country_union, max_move_m):
    """
    If point is not strictly inside country_union, try to move it to the
    nearest interior point. Only move if distance <= max_move_m.
    Returns (new_point, was_moved) where new_point is None if the point
    is outside and cannot be moved within max_move_m.
    """
    if country_union.contains(point):
        return point, False

    # Nearest point on the boundary of the united countries
    nearest_boundary = nearest_points(point, country_union)[1]

    # Direction from boundary point to original point (outward)
    dx = point.x - nearest_boundary.x
    dy = point.y - nearest_boundary.y
    dist = (dx**2 + dy**2) ** 0.5

    if dist == 0:
        # Point is exactly on the boundary – get a guaranteed interior point
        candidate = country_union.representative_point()
    else:
        # Inward direction
        inward_x = -dx / dist
        inward_y = -dy / dist
        epsilon = 1.0  # 1 metre inside
        candidate = Point(
            nearest_boundary.x + inward_x * epsilon,
            nearest_boundary.y + inward_y * epsilon
        )

    # Fallback if the candidate is still not strictly inside
    if not country_union.contains(candidate):
        candidate = country_union.representative_point()

    move_dist = point.distance(candidate)
    if move_dist <= max_move_m:
        return candidate, True
    else:
        # Discard this point – return None to mark it for removal
        return None, False


def main():
    # 1. Load and merge reseller layers, deduplicate exact points
    print("Reading reseller layers...")
    reseller1 = gpd.read_file(PATHS["reseller1"])
    reseller2 = gpd.read_file(PATHS["reseller2"])
    reseller = gpd.GeoDataFrame(pd.concat([reseller1, reseller2], ignore_index=True),
                                crs=reseller1.crs)
    # Deduplicate: keep only one point for each exact location
    # Use WKB representation for robust comparison
    reseller['_wkb'] = reseller.geometry.apply(lambda g: g.wkb)
    reseller = reseller.drop_duplicates(subset='_wkb').drop(columns='_wkb')
    print(f"  After deduplication: {len(reseller)} reseller points")

    # 2. Load and merge filling layers, deduplicate exact points
    print("Reading filling layers...")
    filling1 = gpd.read_file(PATHS["filling1"])
    filling2 = gpd.read_file(PATHS["filling2"])
    filling = gpd.GeoDataFrame(pd.concat([filling1, filling2], ignore_index=True),
                               crs=filling1.crs)
    filling['_wkb'] = filling.geometry.apply(lambda g: g.wkb)
    filling = filling.drop_duplicates(subset='_wkb').drop(columns='_wkb')
    print(f"  After deduplication: {len(filling)} filling points")

    # 3. Load country boundaries and reproject to EPSG:3857 if needed
    print("Loading country boundaries...")
    countries = gpd.read_file(PATHS["countries"])
    if countries.crs is None:
        print("Warning: country shapefile has no CRS – assuming EPSG:4326")
        countries = countries.set_crs("EPSG:4326")
    if countries.crs.to_epsg() != 3857:
        print(f"Reprojecting countries from {countries.crs} to EPSG:3857")
        countries = countries.to_crs("EPSG:3857")
    countries.geometry = countries.geometry.make_valid()
    country_union = unary_union(countries.geometry)
    max_move_m = MAX_MOVE_KM * 1000.0

    # 4. Process both layers (move inside, discard far outliers)
    for name, gdf in [("reseller_africa", reseller), ("filling_africa", filling)]:
        print(f"Processing layer '{name}' ({len(gdf)} points)...")
        new_geoms = []
        moved_count = 0
        removed_count = 0
        for idx, row in gdf.iterrows():
            orig_geom = row.geometry
            if orig_geom is None or not isinstance(orig_geom, Point):
                new_geoms.append(orig_geom)   # keep as is (or could remove)
                continue
            new_geom, moved = move_inside(orig_geom, country_union, max_move_m)
            if new_geom is None:
                # Point was outside and too far – mark for removal
                new_geoms.append(None)
                removed_count += 1
            else:
                new_geoms.append(new_geom)
                if moved:
                    moved_count += 1
        gdf.geometry = new_geoms

        # Remove points that were set to None (too far outside)
        gdf = gdf[gdf.geometry.notnull()].copy()
        gdf = gdf[gdf.is_valid]
        print(f"  Moved {moved_count} points inside, removed {removed_count} far points.")
        print(f"  Remaining: {len(gdf)} points.")

        # Update the variable in the outer scope so we can work with it later
        if name == "reseller_africa":
            reseller = gdf
        else:
            filling = gdf

    # 5. Avoid perfect overlap between reseller and filling points
    # If a filling point coincides with a reseller point, shift it 2 m east.
    print("Separating coincident points between reseller and filling layers...")
    reseller_coords = set()
    for geom in reseller.geometry:
        reseller_coords.add((geom.x, geom.y))   # exact float equality, fine for identical points

    new_filling_geoms = []
    shift_count = 0
    for geom in filling.geometry:
        pt = (geom.x, geom.y)
        if pt in reseller_coords:
            # Shift 2 m east (EPSG:3857, metres)
            new_geom = Point(geom.x + SEPARATION_M, geom.y)
            new_filling_geoms.append(new_geom)
            shift_count += 1
        else:
            new_filling_geoms.append(geom)
    filling.geometry = new_filling_geoms
    print(f"  Shifted {shift_count} filling points to avoid overlap.")

    # 6. Write output layers
    print(f"Writing layer 'reseller_africa' to {PATHS['output']}")
    reseller.to_file(PATHS["output"], layer="reseller_africa", driver="GPKG")

    print(f"Writing layer 'filling_africa' to {PATHS['output']}")
    filling.to_file(PATHS["output"], layer="filling_africa", driver="GPKG")

    print("Done.")


if __name__ == "__main__":
    main()

Using current working directory as base: c:\Users\matti\Desktop\SSA
Reading reseller layers...
  After deduplication: 2631 reseller points
Reading filling layers...
  After deduplication: 1357 filling points
Loading country boundaries...
Reprojecting countries from EPSG:4326 to EPSG:3857
Processing layer 'reseller_africa' (2631 points)...
  Moved 29 points inside, removed 93 far points.
  Remaining: 2538 points.
Processing layer 'filling_africa' (1357 points)...
  Moved 14 points inside, removed 118 far points.
  Remaining: 1239 points.
Separating coincident points between reseller and filling layers...
  Shifted 5 filling points to avoid overlap.
Writing layer 'reseller_africa' to c:\Users\matti\Desktop\SSA\SSA/infrastructure.gpkg
Writing layer 'filling_africa' to c:\Users\matti\Desktop\SSA\SSA/infrastructure.gpkg
Done.
